# Olist E-Ticaret Analitiği — Brezilya Pazarı İlk Bağımsız Analiz

**Amaç:** Olist açık veri setini kullanarak müşteri, sipariş, ürün ve ödeme
verileri arasındaki ilişkileri incelemek; istatistiksel testler ve A/B testleriyle
içgörüler üretmek.

## İçindekiler
1. Kurulum & Veri Yükleme
2. Veri Yapısını İnceleme (her tablonun tanıtımı)
3. Veri Temizleme (eksik değerler, tip dönüşümleri)
4. Tabloları Birleştirme (merge stratejisi)
5. Keşifsel Analiz (EDA)
6. İstatistiksel Testler & A/B Testi
7. Görselleştirme
8. Sonuç & Öneriler

## 1. KURULUM & VERİ YÜKLEME

Bu bölümde çalışma ortamı hazırlanmış ve Olist veri seti içeri aktarılmıştır.

**Yapılan işlemler:**
- Google Drive bağlandı (`drive.mount`), veri seti Drive üzerindeki
  `Olist - Python` klasöründen okunmuştur.
- Gerekli kütüphaneler yüklendi: `pandas`, `numpy`, `plotly.express`.
- Veri seti 7 ayrı CSV dosyasından oluşmaktadır ve her biri kendi
  DataFrame'ine (`df_customers`, `df_orders`, `df_order_items`,
  `df_products`, `df_sellers`, `df_payments`, `df_reviews`) yüklenmiştir.

**Veri setinin genel yapısı:**

| Tablo | Satır Sayısı | Temsil Ettiği Birim |
|---|---|---|
| customers | 99,441 | Her satır bir müşteri |
| orders | 99,441 | Her satır bir sipariş |
| order_items | 112,650 | Her satır bir sipariş kalemi |
| products | 32,951 | Her satır bir ürün |
| sellers | 3,095 | Her satır bir satıcı |
| payments | 103,886 | Her satır bir ödeme kaydı |
| reviews | 99,224 | Her satır bir değerlendirme |

`orders` tablosu, `order_id` ve `customer_id` üzerinden diğer tüm
tablolara bağlanan merkezi tablodur.

In [ ]:
# 1. Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 2. Kütüphaneleri yükle
import pandas as pd
import numpy as np
import plotly.express as px

# 3. Veriyi oku
base_path = "/content/drive/MyDrive/Olist - Python/"

df_customers = pd.read_csv(base_path + "olist_customers_dataset.csv")
df_orders = pd.read_csv(base_path + "olist_orders_dataset.csv")
df_order_items = pd.read_csv(base_path + "olist_order_items_dataset.csv")
df_products = pd.read_csv(base_path + "olist_products_dataset.csv")
df_sellers = pd.read_csv(base_path + "olist_sellers_dataset.csv")
df_payments = pd.read_csv(base_path + "olist_order_payments_dataset.csv")
df_reviews = pd.read_csv(base_path + "olist_order_reviews_dataset.csv")

## 2. VERİ YAPISINI İNCELEME

Yedi tablo (`customers`, `orders`, `order_items`, `products`, `sellers`,
`payments`, `reviews`) `.info()` ve `.isna().sum()` ile incelenmiştir.
Amaç, her tablonun sütun yapısını, veri tiplerini ve eksik değerlerin
nerede yoğunlaştığını görmektir.

**Öne çıkan bulgular:**
- Tarih sütunlarının tamamı (`orders`, `reviews` içinde) başlangıçta
  `object` (metin) türündedir — bir sonraki adımda `datetime`'a dönüştürülecektir.
- `orders` tablosundaki teslimat tarihlerinde (`order_approved_at`,
  `order_delivered_carrier_date`, `order_delivered_customer_date`) eksik
  değerler bulunmaktadır; bu, siparişlerin henüz tamamlanmamış/iptal
  edilmiş olmasından kaynaklanan **doğal** bir eksikliktir, veri hatası değildir.
- `products` tablosunda 610 satırda kategori ve ürün metni bilgisi,
  2 satırda ise boyut/ağırlık bilgisi eksiktir.
- `reviews` tablosunda yorum metni sütunları büyük oranda boştur
  (yorum yazmak opsiyoneldir); `review_score` ise tamamen doludur.

In [ ]:
# tablolar sözlüğü oluştur
tablolar = {
    "customers": df_customers,
    "orders": df_orders,
    "order_items": df_order_items,
    "products": df_products,
    "sellers": df_sellers,
    "payments": df_payments,
    "reviews": df_reviews
}

# tablolara genel bakış: key-value döngüsü, tablo halinde gösterim
for isim, df in tablolar.items():
    print(f"=== {isim} ===")
    print("Shape:", df.shape)
    display(df.head(3))
    print()

=== customers ===
Shape: (99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



=== orders ===
Shape: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



=== order_items ===
Shape: (112650, 7)


...

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ



=== payments ===
Shape: (103886, 5)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71



=== reviews ===
Shape: (99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


In [ ]:
# sadece başlıkları incele (merge)
for isim, df in tablolar.items():
    print(f"=== {isim} ===")
    print(list(df.columns))
    print()

=== customers ===
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

=== orders ===
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

=== order_items ===
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

=== products ===
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

=== sellers ===
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

=== payments ===
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

=== reviews ===
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', '

In [ ]:
# sütun yapılarını incele
for isim, df in tablolar.items():
    print(f"=== {isim} ===")
    print(df.info())
    print()

# her tabloda kaç eksik var, hangi sütunda
for isim, df in tablolar.items():
    print(f"=== {isim} — eksik değerler ===")
    print(df.isna().sum())
    print()

=== customers ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
None

=== orders ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_ti

## 3. VERİYİ TEMİZLEME

**Yapılan işlemler:**
- `orders` ve `reviews` tablolarındaki tüm tarih sütunları `pd.to_datetime()`
  ile gerçek tarih türüne çevrilmiştir.
- `products` tablosunda boyut/ağırlık sütunlarındaki (2 satır) eksik değerler
  medyan ile doldurulmuştur.
- `product_category_name` sütunundaki eksik değerler `"unknown"` etiketiyle
  doldurulmuştur.

**Bilinçli olarak dokunulmayan alanlar:**
- `orders` tablosundaki teslimat tarihi eksiklikleri **doldurulmamıştır** —
  bu boşluklar "henüz teslim edilmedi/iptal edildi" bilgisini taşıdığı için
  yapay bir tarihle doldurmak yanıltıcı olurdu.
- `product_name_lenght`, `product_description_lenght`, `product_photos_qty`
  şu aşamada ana inceleme parametrelerinden değildir (610 satırda eksik).
  İhtiyaç halinde temizlenip analize dahil edilecektir.
- `reviews` tablosundaki yorum metni sütunları (title/message) analiz
  kapsamı dışında tutulmuştur; eksiklik doğal ve beklenen bir durumdur.

In [ ]:
# 1. Tarihleri gerçek datetime'a çevir
date_cols_orders = ["order_purchase_timestamp", "order_approved_at",
                    "order_delivered_carrier_date", "order_delivered_customer_date",
                    "order_estimated_delivery_date"]
for col in date_cols_orders:
    df_orders[col] = pd.to_datetime(df_orders[col])

date_cols_reviews = ["review_creation_date", "review_answer_timestamp"]
for col in date_cols_reviews:
    df_reviews[col] = pd.to_datetime(df_reviews[col])

# 2. Products — az sayıdaki boyut/ağırlık eksiğini medyan ile doldur
boyut_sutunlari = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
for col in boyut_sutunlari:
    df_products[col] = df_products[col].fillna(df_products[col].median())

# 3. Kategori eksik olan satırları etiketle (silmek yerine işaretle)
df_products["product_category_name"] = df_products["product_category_name"].fillna("unknown")

In [ ]:
# KONTROL
df_orders.info()  # tarih = datetime
df_products.isna().sum()  # sadece kategoriyle ilgili eksikler "unknown" oldu, sayısal eksikler gitti

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


,0
product_id,0
product_category_name,0
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,0
product_length_cm,0
product_height_cm,0
product_width_cm,0


## 4. TABLOLARI BİRLEŞTİRME (Merge)

Tüm tablolar ortak kimlik sütunları üzerinden tek bir ana tabloda birleştirilecektir:

- `orders` ↔ `order_items` → **order_id**
- `orders` ↔ `payments` → **order_id**
- `orders` ↔ `reviews` → **order_id**
- `orders` ↔ `customers` → **customer_id**
- `order_items` ↔ `products` → **product_id**
- `order_items` ↔ `sellers` → **seller_id**

Sonuçta oluşan `df_main` tablosu, sipariş + ürün + ödeme + müşteri + memnuniyet
bilgisini bir arada tutar ve keşifsel analiz bu tablo üzerinden yürütülecektir.

**Not:** `order_items` ve `payments` tablolarında bir siparişe ait birden fazla
satır olabildiğinden (çoklu ürün ya da parçalı ödeme), birleştirme sonrası bazı
sipariş kimlikleri

In [ ]:
# orders'ı merkezli tablo oluştur

# 1. orders + order_items (sipariş + kalemler — fiyat, ürün, satıcı bilgisi gelir)
df_main = df_orders.merge(df_order_items, on="order_id", how="left")

# 2. + payments (ödeme bilgisi)
df_main = df_main.merge(df_payments, on="order_id", how="left")

# 3. + reviews (memnuniyet puanı)
df_main = df_main.merge(df_reviews[["order_id", "review_score"]], on="order_id", how="left")

# 4. + customers (müşterinin konumu gibi bilgiler, istersen)
df_main = df_main.merge(df_customers, on="customer_id", how="left")

print(df_main.shape)
df_main.head()

(119143, 23)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,freight_value,payment_sequential,payment_type,payment_installments,payment_value,review_score,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,8.72,1.0,credit_card,1.0,18.12,4.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,8.72,3.0,voucher,1.0,2.00,4.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,8.72,2.0,voucher,1.0,18.59,4.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,595fac2a385ac33a80bd5114aec74eb8,...,22.76,1.0,boleto,1.0,141.46,4.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,aa4383b373c6aca5d8797843e5594415,...,19.22,1.0,credit_card,3.0,179.12,5.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO


## 5. KEŞİFSEL ANALİZ (EDA)

Bu bölümde üç eksende inceleme yapılacaktır:

**Ürün Ekseni**
- Kaç farklı ürün satılmış, en çok satan ürünler (adet bazında)
- En çok gelir/marj getiren ürünler
- Bu ürünlerin review puanlarıyla ilişkisi

**Ödeme Ekseni**
- En çok kullanılan ödeme yöntemi
- Ödeme yöntemi / fiyat ile review puanı arasındaki olası ilişki

**Satıcı Ekseni**
- Top 5-10 satıcı (satış/gelir bazında)
- Bu satıcıların ürünleri, en çok satan ürünlerle örtüşüyor mu
- Satıcı performansı ve olası iyileştirme alanları

Bu bölümdeki bulgular, bir sonraki "İstatistiksel Testler" bölümünde
hipotez olarak test edilecektir.

In [ ]:
# ÜRÜN EKSENİ

# En çok satan ürünler (kaç farklı siparişte yer almış — adet bazlı)
en_cok_satan = df_main.groupby("product_id")["order_id"].nunique().sort_values(ascending=False)
print("En çok satan 10 ürün:")
display(en_cok_satan.head(10))

# En çok gelir getiren ürünler (toplam price bazında)
# Önce order_item bazında tekilleştiriyoruz (çoklu ödeme satırları yüzünden çifte sayım olmasın diye)
df_urun_gelir = df_main.drop_duplicates(subset=["order_id", "order_item_id"])
en_cok_gelir = df_urun_gelir.groupby("product_id")["price"].sum().sort_values(ascending=False)
print("En çok gelir getiren 10 ürün:")
display(en_cok_gelir.head(10))

En çok satan 10 ürün:


,order_id
product_id,
99a4788cb24856965c36a24e339b6058,467
aca2eb7d00ea1a7b8ebd4e68314663af,431
422879e10f46682990de24d770e7f83d,352
d1c427060a0f73f6b889a5c7c61f2ac4,323
389d119b48cf3043d311335e499d9c6b,311
53b36df67ebb7c41585e8d54d6772e08,306
368c6c730842d78016ad823897a372db,291
53759a2ecddad2bb87a079a1f1519f73,287
154e7e31ebfa092203795c972e5804a6,269


En çok gelir getiren 10 ürün:


,price
product_id,
bb50f2e236e5eea0100680137654686c,63885.00
6cdd53843498f92890544667809f1595,54730.20
d6160fb7873f184099d9bc95e30376af,48899.34
d1c427060a0f73f6b889a5c7c61f2ac4,47214.51
99a4788cb24856965c36a24e339b6058,43025.56
3dd2a17168ec895c781a9191c1e95ad7,41082.60
25c38557cf793876c5abdd5931f922db,38907.32
5f504b3a1c75b73d6151be81eb05bdc9,37733.90
53b36df67ebb7c41585e8d54d6772e08,37683.42


**Gözlem:** En çok satan 10 ürün ile en çok gelir getiren 10 ürün büyük ölçüde
farklı listeler. En popüler ürün (`99a4788cb...`, 467 sipariş) gelir sıralamasında
5. sırada, en çok gelir getiren ürün (`bb50f2e23...`) ise popülerlik listesinde
ilk 10'da bile yer almıyor. Bu, en çok satan ürünlerin muhtemelen düşük birim
fiyatlı, en çok gelir getirenlerin ise yüksek birim fiyatlı ama az sayıda satılan
ürünler olabileceğini düşündürüyor.

Bu hipotezi doğrulamak için, her iki gruptaki ürünlerin ortalama birim
fiyatlarını incelemeyi uygun gördüm.

In [ ]:
en_cok_satan_id = en_cok_satan.index[0]
en_cok_gelir_id = en_cok_gelir.index[0]

fiyat_satan = df_urun_gelir[df_urun_gelir["product_id"] == en_cok_satan_id]["price"].mean()
fiyat_gelir = df_urun_gelir[df_urun_gelir["product_id"] == en_cok_gelir_id]["price"].mean()

print(f"En çok satan ürünün ortalama fiyatı: {fiyat_satan:.2f}")
print(f"En çok gelir getiren ürünün ortalama fiyatı: {fiyat_gelir:.2f}")

En çok satan ürünün ortalama fiyatı: 88.17
En çok gelir getiren ürünün ortalama fiyatı: 327.62


**Sonuç:** Hipotezimiz doğrulandı. En çok satan ürünün ortalama fiyatı 88.17
iken, en çok gelir getiren ürünün ortalama fiyatı 327.62 — yaklaşık 3.7 kat
daha yüksek. Bu, platformda iki farklı ürün profilinin bulunduğunu gösteriyor:

- **Hacim odaklı ürünler:** Düşük birim fiyat, yüksek satış adedi (örn. en çok
  satan ürün) — geniş kitleye ulaşan, sık tekrarlanan alışverişler.
- **Değer odaklı ürünler:** Yüksek birim fiyat, düşük satış adedi (örn. en çok
  gelir getiren ürün) — az sayıda ama yüksek tutarlı işlemler.

**İş çıkarımı:** Envanter ve pazarlama stratejisi bu iki grup için ayrı ayrı
değerlendirilmeli — hacim odaklı ürünlerde erişim/görünürlük, değer odaklı
ürünlerde ise dönüşüm oranı ve müşteri güveni (yüksek tutarlı alışverişte
karar süreci daha uzundur) öncelikli olabilir.

Ürün ekseninde satış/gelir performansı ile müşteri memnuniyeti arasında bir
tutarsızlık olup olmadığını görmek amacıyla, en çok satan ve en çok gelir
getiren ürünlerin review puanları karşılaştırılmıştır.

In [ ]:
print("En çok satan 10 ürün ve review puanları (satış hacmine göre sıralı):")
satan_review = df_main[df_main["product_id"].isin(top10_satan_id)].groupby("product_id")["review_score"].mean()
satan_review_sirali = satan_review.reindex(top10_satan_id)  # en_cok_satan'daki sırayı koru
satan_review_df = satan_review_sirali.reset_index()
satan_review_df.columns = ["product_id", "review_score"]
satan_review_df["siparis_sayisi"] = en_cok_satan.head(10).values
display(satan_review_df)

print("\nEn çok gelir getiren 10 ürün ve review puanları (gelire göre sıralı):")
gelir_review = df_main[df_main["product_id"].isin(top10_gelir_id)].groupby("product_id")["review_score"].mean()
gelir_review_sirali = gelir_review.reindex(top10_gelir_id)  # en_cok_gelir'daki sırayı koru
gelir_review_df = gelir_review_sirali.reset_index()
gelir_review_df.columns = ["product_id", "review_score"]
gelir_review_df["toplam_gelir"] = en_cok_gelir.head(10).values
display(gelir_review_df)

En çok satan 10 ürün ve review puanları (satış hacmine göre sıralı):


,product_id,review_score,siparis_sayisi
0,99a4788cb24856965c36a24e339b6058,3.914894,467
1,aca2eb7d00ea1a7b8ebd4e68314663af,4.020638,431
2,422879e10f46682990de24d770e7f83d,3.927022,352
3,d1c427060a0f73f6b889a5c7c61f2ac4,4.096045,323
4,389d119b48cf3043d311335e499d9c6b,4.106173,311
5,53b36df67ebb7c41585e8d54d6772e08,4.200617,306
6,368c6c730842d78016ad823897a372db,3.908861,291
7,53759a2ecddad2bb87a079a1f1519f73,3.884319,287
8,154e7e31ebfa092203795c972e5804a6,4.319728,269
9,2b4609f8948be18874494203496bc318,4.087273,259



En çok gelir getiren 10 ürün ve review puanları (gelire göre sıralı):


,product_id,review_score,toplam_gelir
0,bb50f2e236e5eea0100680137654686c,4.219048,63885.00
1,6cdd53843498f92890544667809f1595,4.322785,54730.20
2,d6160fb7873f184099d9bc95e30376af,4.512195,48899.34
3,d1c427060a0f73f6b889a5c7c61f2ac4,4.096045,47214.51
4,99a4788cb24856965c36a24e339b6058,3.914894,43025.56
5,3dd2a17168ec895c781a9191c1e95ad7,4.206522,41082.60
6,25c38557cf793876c5abdd5931f922db,2.581395,38907.32
7,5f504b3a1c75b73d6151be81eb05bdc9,4.555556,37733.90
8,53b36df67ebb7c41585e8d54d6772e08,4.200617,37683.42
9,aca2eb7d00ea1a7b8ebd4e68314663af,4.020638,37608.90


**Gözlem:** En çok satan 10 ürünün review puanları dar bir aralıkta
(3.88–4.32) toplanıyor — bu ürünler tutarlı bir müşteri deneyimi sunuyor.

En çok gelir getiren 10 ürüne bakıldığında ise puanlar çok daha geniş bir
aralığa yayılıyor (2.58–4.55). Özellikle **gelir sıralamasındaki 7. ürün**,
38,907.32 ile önemli bir gelir kaynağı olmasına rağmen sadece **2.58**
review puanı almış — listedeki en düşük puan, ve genel ortalamanın belirgin
şekilde altında.

**Risk değerlendirmesi:** Bu tek ürün, hem yüksek gelir getiren hem de ciddi
memnuniyet sorunu taşıyan bir "kırmızı bayrak" niteliğinde. Diğer dokuz ürün
3.9–4.5 bandında toplanmışken, bu ürünün tek başına düşük kalması, genel bir
kategori sorunu değil, **bu ürüne özgü** bir sorun (teslimat, kalite, ya da
fiyat/değer algısı) olduğuna işaret ediyor. Bu ürünün detaylı incelenmesi
(teslimat süreleri, olası şikayet nedenleri) önerilir.

Gelir sıralamasındaki 7. ürünün neden bu kadar düşük review puanı aldığını
anlamak için, elimizdeki teslimat detaylarını (gerçek teslimat tarihi vs.
tahmini teslimat tarihi) inceleyerek olası bir gecikme sorunu olup olmadığına
bakmayı uygun gördüm.

In [ ]:
urun_7 = "25c38557cf793876c5abdd5931f922db"

detay = df_main[df_main["product_id"] == urun_7][
    ["order_id", "price", "review_score", "order_purchase_timestamp",
     "order_delivered_customer_date", "order_estimated_delivery_date"]
].drop_duplicates(subset="order_id")

detay["gecikme_gun"] = (detay["order_delivered_customer_date"] - detay["order_estimated_delivery_date"]).dt.days

display(detay)
print("\nOrtalama gecikme (gün):", detay["gecikme_gun"].mean())
print("Kaç siparişte gecikme var (pozitif gün):", (detay["gecikme_gun"] > 0).sum(), "/", len(detay))

,order_id,price,review_score,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,gecikme_gun
13668,d17c532b7b6e2aa56945b55f0757fee4,987.00,2.0,2018-03-25 21:22:34,2018-04-06 17:18:42,2018-04-17,-11
14054,305c64156d718f215d3d7cdc8be8b39c,1106.99,1.0,2018-04-24 10:58:24,2018-05-03 20:19:14,2018-05-29,-26
14251,708d3d7c0c996a92a48a8cf501f6fd8b,1089.00,5.0,2018-01-17 14:35:51,2018-02-09 20:38:38,2018-02-08,1
22514,0a9c633395c19d5fa38b9e5313db37a6,999.00,NaN,2018-03-01 21:21:22,2018-03-21 01:51:55,2018-03-22,-1
22861,925ee6cfbb7a083de5384773197aa9f6,1106.99,1.0,2018-04-08 19:22:37,2018-05-02 18:32:03,2018-05-07,-5
23572,6636442daffc5438b5f2ff1f068d449b,1106.99,1.0,2018-04-13 00:17:20,2018-04-30 21:22:26,2018-05-11,-11
24013,18e9c9a8bc880b4b3aeafede031f7587,999.00,1.0,2018-03-13 09:52:21,2018-04-06 23:38:37,2018-04-17,-11
25212,f2e66c1285248a57d6f934463632274f,1049.00,2.0,2018-02-28 19:55:39,2018-03-14 19:08:51,2018-03-22,-8
28967,d2733958b49450e568a7cda310ac055b,987.00,4.0,2018-03-24 19:27:39,2018-04-10 19:20:53,2018-04-16,-6
30524,d61c57ca1a55ced6197d4f0c6da594e8,949.90,1.0,2018-03-16 15:23:18,2018-04-16 20:11:50,2018-04-09,7



Ortalama gecikme (gün): -6.447368421052632
Kaç siparişte gecikme var (pozitif gün): 8 / 38


**Sonuç:** Teslimat tarihleri incelendiğinde, bu ürünün siparişlerinin
büyük çoğunluğunun tahmin edilen tarihten **önce** (erken) teslim edildiği
görülmüştür. Ancak erken teslim edilen birçok siparişte bile düşük review
puanları (1-2 yıldız) alınmış olması, **teslimat gecikmesinin düşük puanların
ana sebebi olmadığını** göstermektedir.

Ürünün fiyat aralığının (~950-1100 TL) platform ortalamasına göre yüksek
olduğu görülmektedir — bu, düşük puanların teslimat süresinden çok, ürün
kalitesi ya da fiyat/değer algısıyla ilişkili olabileceğine işaret etmektedir.
Bu veri setinde ürün kalitesini doğrudan ölçen bir değişken bulunmadığından,
kesin bir nedensellik ilişkisi kurulamamaktadır; ancak bu bulgu, ilgili
ürünün detaylı incelenmesi (müşteri yorumları, ürün özellikleri) için bir
öneri olarak değerlendirilebilir.

Ürün ekseni analizini tamamlamak amacıyla, bu kez tersinden bir soruya
baktık: en yüksek review puanına sahip ürünler, aynı zamanda en çok satan
ya da en çok gelir getiren ürünler arasında yer alıyor mu? Bu, memnuniyetin
ticari başarıyla ne ölçüde örtüştüğünü görmek için incelenmiştir.

In [ ]:
en_yuksek_puan = df_main.groupby("product_id")["review_score"].mean().sort_values(ascending=False).head(10)

ortak_satan = set(en_yuksek_puan.index) & set(top10_satan_id)
ortak_gelir = set(en_yuksek_puan.index) & set(top10_gelir_id)

print("En yüksek puanlı 10 üründen, en çok satanlarla ortak olan sayısı:", len(ortak_satan))
print("En yüksek puanlı 10 üründen, en çok gelir getirenle ortak olan sayısı:", len(ortak_gelir))

En yüksek puanlı 10 üründen, en çok satanlarla ortak olan sayısı: 0
En yüksek puanlı 10 üründen, en çok gelir getirenle ortak olan sayısı: 0


**Sonuç:** En yüksek review puanına sahip 10 üründen **hiçbiri**, ne en çok
satan ne de en çok gelir getiren ürünler arasında yer almıyor (kesişim: 0/10
her iki grup için de).

**Fırsat değerlendirmesi:** Bu, platform için önemli bir büyüme fırsatına
işaret ediyor. Müşterileri en çok memnun eden ürünler, şu anda ticari olarak
en çok öne çıkan ürünler değil — yani platformun "en iyi" ürünleri, satış
hacmi ya da gelir açısından henüz yeterince görünür/tercih edilir hale
gelmemiş. Pazarlama görünürlüğü (öne çıkarma, önerilen ürünler listesi),
fiyatlandırma stratejisi ya da arama/sıralama algoritmalarında bu yüksek
puanlı ürünlere öncelik verilmesi, hem satışları hem de genel müşteri
memnuniyetini birlikte artırabilecek düşük riskli bir aksiyon alanı olarak
değerlendirilebilir.

In [ ]:
# en yüksek puanlı ürünler sıralaması
print("En yüksek review puanına sahip 10 ürün:")
display(en_yuksek_puan)

En yüksek review puanına sahip 10 ürün:


,review_score
product_id,
000b8f95fcb9e0096488278317764d19,5.0
fffdb2d0ec8d6a61f0a0a0db3f25b441,5.0
00066f42aeeb9f3007548bb9d3f33c38,5.0
fff9553ac224cec9d15d49f5a263411f,5.0
ffd9ac56db9194a413298faaa03cd176,5.0
ffd7628b0b0b98ebc549e8e4c54a59af,5.0
ffd63ee42a5c8cc5a15a1c8e2aa50011,5.0
000d9be29b5207b54e86aa1b1ac54872,5.0
8aebbc3445bef4a356ae41d226a7b5da,5.0


**Not:** Bu 10 ürünün fiyat ve teslimat süresi gibi ek özellikleri, zaman
kısıtı nedeniyle bu analiz kapsamında incelenmemiştir; ileri bir çalışmada
ele alınabilir.

**Ürün Ekseni Özeti:**
1. En çok satan ile en çok gelir getiren ürünler farklı — hacim odaklı ve
   değer odaklı iki ayrı ürün profili mevcut.
2. Gelir sıralamasındaki 7. ürün, yüksek gelirine rağmen düşük memnuniyet
   (2.58 puan) taşıyor; sebep teslimat gecikmesi değil, muhtemelen ürün
   kalitesi/fiyat algısı.
3. En yüksek puanlı 10 ürün, satış/gelir liderleriyle hiç örtüşmüyor —
   bu, memnuniyeti yüksek ama ticari olarak az görünür ürünlerde büyük
   bir büyüme fırsatı olduğunu gösteriyor.

## Ödeme Ekseni

Bu eksende, müşterilerin ödeme davranışlarını ve bunun memnuniyetle
ilişkisini inceliyoruz:
- En çok kullanılan ödeme yöntemi hangisi?
- Ödeme yöntemi ile review puanı arasında bir ilişki var mı?
- Ödeme yöntemi ile sipariş tutarı arasında bir ilişki var mı?

Bu bulgular, bir sonraki bölümde istatistiksel testlerle (ANOVA/Kruskal-Wallis)
doğrulanacaktır.

İlk olarak, müşterilerin en çok hangi ödeme yöntemini tercih ettiğine
bakıyoruz.

In [ ]:
odeme_dagilimi = df_payments["payment_type"].value_counts()
print("Ödeme yöntemi dağılımı:")
display(odeme_dagilimi)

print("\nYüzdesel dağılım:")
display(df_payments["payment_type"].value_counts(normalize=True) * 100)

Ödeme yöntemi dağılımı:


,count
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3



Yüzdesel dağılım:


,proportion
payment_type,
credit_card,73.922376
boleto,19.043952
voucher,5.558978
debit_card,1.471806
not_defined,0.002888


Ödeme yönteminin müşteri memnuniyeti ve harcama tutarıyla ilişkili olup
olmadığını görmek için, her ödeme tipinin ortalama review puanı ve ortalama
ödeme tutarını inceliyoruz.

In [ ]:
odeme_analiz = df_main.groupby("payment_type").agg({
    "review_score": "mean",
    "payment_value": "mean"
}).round(2)

odeme_analiz.columns = ["ortalama_review_puani", "ortalama_odeme_tutari"]
odeme_analiz = odeme_analiz.sort_values("ortalama_odeme_tutari", ascending=False)

display(odeme_analiz)

,ortalama_review_puani,ortalama_odeme_tutari
payment_type,,
credit_card,4.02,179.72
boleto,4.01,177.27
debit_card,4.15,150.86
voucher,3.96,67.43
not_defined,1.67,0.00


**Gözlem:** Ödemelerin büyük çoğunluğu (%73.9) kredi kartı ile yapılmaktadır
ve kredi kartı/boleto ile yapılan ödemeler, banka kartı/voucher'dan belirgin
şekilde yüksek tutarludur.

**Olası açıklama:** Kredi kartının sunduğu taksit imkanı, müşterilerin yüksek
tutarlı alışverişlerde bu yöntemi tercih etmesini teşvik ediyor olabilir
[korelasyon katsayısı: X]. Bu doğruysa, taksit seçeneğinin checkout sürecinde
daha görünür/teşvik edici şekilde sunulması (örn. "12 taksitle X TL" gibi
vurgular), ortalama sepet tutarını artırabilir — memnuniyet ödeme yönteminden
bağımsız olduğu için bu, müşteri deneyimini riske atmadan uygulanabilecek
düşük riskli bir öneridir.

In [ ]:
# korelasyon katsayısı
print("Taksit sayısı ile ödeme tutarı arasındaki korelasyon:")
korelasyon = df_main["payment_installments"].corr(df_main["payment_value"])
print(korelasyon)

Taksit sayısı ile ödeme tutarı arasındaki korelasyon:
0.2736468683558472


**Sonuç:** Taksit sayısı ile ödeme tutarı arasında zayıf-orta düzeyde
(r = 0.27) pozitif bir korelasyon gözlemlenmiştir. Bu, taksit imkanının
yüksek tutarlı alışverişi bir miktar teşvik ediyor olabileceğine dair bir
ipucu sunmaktadır; ancak ilişkinin istatistiksel anlamlılığı bu aşamada
resmi olarak test edilmemiştir (zaman kısıtı nedeniyle EDA kapsamında
bırakılmıştır).

**İş çıkarımı:** Taksit seçeneğinin checkout sürecinde daha görünür
kılınması, ortalama sepet tutarını artırabilecek düşük riskli bir aksiyon
olarak değerlendirilebilir; ancak korelasyonun zayıf-orta düzeyde olması,
tutarı belirleyen başka faktörlerin (ürün kategorisi, müşteri segmenti gibi)
de devreye girdiğini göstermektedir.

## Satıcı Ekseni

Bu eksende platform üzerindeki satıcıların performansını inceliyoruz:
- Top 5-10 satıcı kimler (satış hacmi/gelir bazında)?
- Bu satıcıların ürünleri, en çok satan/en çok gelir getiren ürünlerle örtüşüyor mu?
- Satıcı bazında ortalama teslimat süresi ve review puanı nasıl?
  (Teslimat performansı satıcı değerlendirmesinin bir parçası olarak ele alınmıştır.)

Bu bulgular, iyileştirme alanlarını ve satıcı iş birliği fırsatlarını
ortaya koymayı amaçlamaktadır.

In [ ]:
# satış hacmine göre top 10
top_satici_hacim = df_urun_gelir.groupby("seller_id")["order_id"].nunique().sort_values(ascending=False)
print("En çok sipariş alan 10 satıcı:")
display(top_satici_hacim.head(10))

# gelire göre top 10
top_satici_gelir = df_urun_gelir.groupby("seller_id")["price"].sum().sort_values(ascending=False)
print("\nEn çok gelir getiren 10 satıcı:")
display(top_satici_gelir.head(10))

En çok sipariş alan 10 satıcı:


,order_id
seller_id,
6560211a19b47992c3666cc44a7e94c0,1854
4a3ca9315b744ce9f8e9374361493884,1806
cc419e0650a3c5ba77189a1882b7556a,1706
1f50f920176fa81dab994f9023523100,1404
da8622b14eb17ae2831f4ac5b9dab84a,1314
955fee9216a65b617aa5c0531780ce60,1287
7a67c85e85bb2ce8582c35f2203ad736,1160
ea8482cd71df3c1969d7b9473ff13abc,1146
4869f7a5dfa277a7dca6462dcf3b52b2,1132



En çok gelir getiren 10 satıcı:


,price
seller_id,
4869f7a5dfa277a7dca6462dcf3b52b2,229472.63
53243585a1d6dc2643021fd1853d8905,222776.05
4a3ca9315b744ce9f8e9374361493884,200472.92
fa1c13f2614d7b5c4749cbc52fecda94,194042.03
7c67e1448b00f6e969d365cea6b010ab,187923.89
7e93a43ef30c4f03f38b393420bc753a,176431.87
da8622b14eb17ae2831f4ac5b9dab84a,160236.57
7a67c85e85bb2ce8582c35f2203ad736,141745.53
1025f0e2d44d7041d6cf58b6550e0bfa,138968.55


**Gözlem:** Satıcı ekseninde, ürün ekseninden farklı olarak, hacim ve gelir
sıralamaları arasında kısmi bir örtüşme var — bazı satıcılar (örn.
`4a3ca9315...`, `da8622b14...`, `7a67c85e8...`) hem en çok sipariş alan hem
de en çok gelir getiren satıcılar arasında yer alıyor. Ancak en çok sipariş
alan satıcı (`6560211a1...`, 1854 sipariş), gelir sıralamasında ilk 10'da
görünmüyor — bu, o satıcının düşük birim fiyatlı ürünlerde yoğunlaştığını
düşündürüyor (ürün ekseninde gözlemlediğimiz "hacim vs değer" ayrımının
satıcı düzeyindeki yansıması).

In [ ]:
# Önce her siparişin teslimat süresini hesaplayalım (satın alma - teslimat arası gün)
df_main["teslimat_suresi"] = (df_main["order_delivered_customer_date"] - df_main["order_purchase_timestamp"]).dt.days

# Top 10 hacimli satıcıların performansı
top10_hacim_id = top_satici_hacim.head(10).index
satici_performans_hacim = df_main[df_main["seller_id"].isin(top10_hacim_id)].groupby("seller_id").agg({
    "teslimat_suresi": "mean",
    "review_score": "mean"
}).round(2)
satici_performans_hacim.columns = ["ort_teslimat_gun", "ort_review_puani"]
print("En çok sipariş alan 10 satıcının performansı:")
display(satici_performans_hacim)

# Top 10 gelirli satıcıların performansı
top10_gelir_id = top_satici_gelir.head(10).index
satici_performans_gelir = df_main[df_main["seller_id"].isin(top10_gelir_id)].groupby("seller_id").agg({
    "teslimat_suresi": "mean",
    "review_score": "mean"
}).round(2)
satici_performans_gelir.columns = ["ort_teslimat_gun", "ort_review_puani"]
print("\nEn çok gelir getiren 10 satıcının performansı:")
display(satici_performans_gelir)

En çok sipariş alan 10 satıcının performansı:


,ort_teslimat_gun,ort_review_puani
seller_id,,
1f50f920176fa81dab994f9023523100,15.17,3.98
3d871de0142ce09b7081e2b9d1733cb1,12.89,4.12
4869f7a5dfa277a7dca6462dcf3b52b2,14.74,4.11
4a3ca9315b744ce9f8e9374361493884,13.80,3.80
6560211a19b47992c3666cc44a7e94c0,9.11,3.91
7a67c85e85bb2ce8582c35f2203ad736,10.56,4.23
955fee9216a65b617aa5c0531780ce60,10.29,4.05
cc419e0650a3c5ba77189a1882b7556a,11.14,4.05
da8622b14eb17ae2831f4ac5b9dab84a,10.65,4.07



En çok gelir getiren 10 satıcının performansı:


,ort_teslimat_gun,ort_review_puani
seller_id,,
1025f0e2d44d7041d6cf58b6550e0bfa,11.52,3.86
4869f7a5dfa277a7dca6462dcf3b52b2,14.74,4.11
4a3ca9315b744ce9f8e9374361493884,13.80,3.80
53243585a1d6dc2643021fd1853d8905,12.85,4.07
7a67c85e85bb2ce8582c35f2203ad736,10.56,4.23
7c67e1448b00f6e969d365cea6b010ab,22.04,3.39
7e93a43ef30c4f03f38b393420bc753a,10.87,4.21
955fee9216a65b617aa5c0531780ce60,10.29,4.05
da8622b14eb17ae2831f4ac5b9dab84a,10.65,4.07


**Gözlem:** Top satıcılar arasında teslimat süresi ile review puanı arasında
gözle görülür bir ilişki var. En yavaş teslimat yapan satıcı (`7c67e1448...`,
ortalama 22.04 gün — listedeki diğer satıcılardan 2 kat daha yavaş), aynı
zamanda listedeki en düşük review puanına (3.39) sahip. Buna karşılık,
hızlı teslimat yapan satıcılar (`7a67c85e8...`, 10.56 gün) daha yüksek
puanlar (4.23) alıyor.

Bu, ürün ekseninde (7. ürün incelemesinde) gecikme ile düşük puan arasında
net bir ilişki bulamamış olmamızla ilginç bir tezat oluşturuyor — bireysel
ürün düzeyinde net olmayan bir örüntü, satıcı düzeyinde (birçok siparişin
ortalaması alındığında) daha belirgin hale geliyor.

**İş çıkarımı:** Teslimat süresi 15 günü aşan satıcılar için lojistik
süreçlerin gözden geçirilmesi önerilir — bu, hem müşteri memnuniyetini hem
de dolaylı olarak satıcının platform üzerindeki performans notunu
iyileştirebilir. Bu ilişkinin istatistiksel anlamlılığı, bir sonraki
bölümde korelasyon testi ile doğrulanabilir.

In [ ]:
# Top satıcıların sattığı ürünler, en yüksek puanlı 10 ürün listesinde mi?
top_satici_urunleri = df_main[df_main["seller_id"].isin(top10_gelir_id)]["product_id"].unique()
ortak = set(en_yuksek_puan.index) & set(top_satici_urunleri)
print("En yüksek puanlı 10 üründen, top satıcıların sattığı ürünlerle ortak olan:", len(ortak))

En yüksek puanlı 10 üründen, top satıcıların sattığı ürünlerle ortak olan: 0


**Bağlantılı Gözlem (Ürün-Satıcı Ekseni):** En yüksek review puanlı 10
üründen hiçbiri (0/10), top satıcıların (gelir bazında ilk 10) sattığı
ürünler arasında yer almıyor. Bu, daha önce gözlemlediğimiz "en yüksek
puanlı ürünler ne en çok satan ne de en çok gelir getiren ürünler arasında"
bulgusuyla birleştiğinde net bir tema ortaya çıkarıyor:

**Platformun en beğenilen ürünleri, mevcut ticari başarı ölçütlerinin
(satış hacmi, gelir, satıcı performansı) hiçbirinde temsil edilmiyor.**

**İş çıkarımı:** Bu, tek seferlik bir tesadüf değil, tutarlı bir örüntü —
platformda memnuniyeti yüksek ama görünürlüğü/erişimi düşük bir ürün
segmenti var. Bu ürünlerin ve onları satan satıcıların belirlenip
öne çıkarılması (öneri algoritmaları, pazarlama, arama sıralaması),
hem müşteri memnuniyetini hem ticari performansı aynı anda artırabilecek,
düşük riskli ve yüksek potansiyelli bir fırsat alanıdır.

## 6. İSTATİSTİKSEL TESTLER

Keşifsel analizde üç eksende (ürün, ödeme, satıcı) gözlemlenen ilişkilerin
istatistiksel anlamlılığı bu bölümde test edilmektedir:

**Ürün Ekseni**
1. Fiyat ile review puanı arasındaki korelasyon (pahalı ürünler daha mı az/çok memnun ediyor?)

**Ödeme Ekseni**

2. Ödeme yöntemi ile review puanı arasında fark var mı? (ANOVA/Kruskal-Wallis)
3. Taksit sayısı ile ödeme tutarı arasındaki korelasyonun anlamlılığı

**Satıcı Ekseni**

4. Teslimat süresi ile review puanı arasındaki korelasyon (genel veri setinde)

Her test için H0/H1 tanımlanacak, alpha = 0.05 eşiği kullanılacaktır.

### Ürün Ekseni: Fiyat ile Review Puanı İlişkisi

Keşifsel analizde, en çok gelir getiren (yüksek fiyatlı) ürünlerin review
puanlarının, en çok satan (düşük fiyatlı) ürünlere kıyasla daha geniş bir
aralığa yayıldığını gözlemlemiştik. Bu gözlemi genelleştirip, tüm veri
setinde fiyat ile review puanı arasında istatistiksel olarak anlamlı bir
ilişki olup olmadığını test ediyoruz.

**H0:** Fiyat ile review puanı arasında ilişki yoktur (korelasyon = 0).
**H1:** Fiyat ile review puanı arasında ilişki vardır.

In [ ]:
from scipy import stats

# Eksik değerleri (varsa) çıkararak korelasyon testi yapalım
veri_temiz = df_urun_gelir[["price", "review_score"]].dropna()

korelasyon, p_value = stats.pearsonr(veri_temiz["price"], veri_temiz["review_score"])

print("Korelasyon katsayısı:", round(korelasyon, 4))
print("P-value:", p_value)

if p_value < 0.05:
    print("H0 reddedilir — fiyat ile review puanı arasında anlamlı bir ilişki var.")
else:
    print("H0 reddedilemez — anlamlı bir ilişki olduğuna dair yeterli kanıt yok.")

Korelasyon katsayısı: -0.0041
P-value: 0.16702880645188262
H0 reddedilemez — anlamlı bir ilişki olduğuna dair yeterli kanıt yok.


**Sonuç:** Fiyat ile review puanı arasında istatistiksel olarak anlamlı bir
ilişki bulunamamıştır (r = -0.004, p = 0.167). Bu, korelasyonun pratik
olarak sıfıra çok yakın olması ve p-value'nun 0.05 eşiğinin üzerinde
kalmasıyla doğrulanmaktadır.

Bu sonuç, EDA'daki gözlemimizi netleştiriyor: en çok gelir getiren (pahalı)
ürünlerdeki geniş puan aralığı, fiyatın **kendisinin** memnuniyeti
etkilediği anlamına gelmiyor. Yani "pahalı ürün = düşük/yüksek puan" gibi
sistematik bir eğilim yok — sadece pahalı ürünlerde puanların dağılımı
(varyansı) daha geniş, bu da muhtemelen az sayıda siparişe dayalı
ortalamaların şansa daha açık olmasından kaynaklanıyor (küçük örneklem
etkisi). 7. ürün gibi tekil vakalar, genel bir "fiyat-memnuniyet"
ilişkisinin kanıtı değil, kendine özgü, ayrı ele alınması gereken
istisnalardır.

### Ödeme Ekseni: Ödeme Yöntemi ile Review Puanı İlişkisi

Keşifsel analizde, farklı ödeme yöntemlerinin (kredi kartı, boleto, banka
kartı, voucher) ortalama review puanlarının birbirine yakın (4.0 civarı)
olduğunu gözlemlemiştik. Bu gözlemi doğrulamak için, ödeme yöntemleri
arasında istatistiksel olarak anlamlı bir puan farkı olup olmadığını
test ediyoruz.

**H0:** Ödeme yöntemleri arasında ortalama review puanı farkı yoktur.
**H1:** En az bir ödeme yönteminin ortalama review puanı diğerlerinden farklıdır.

In [ ]:
from scipy import stats

gruplar = [df_main[df_main["payment_type"] == tip]["review_score"].dropna()
           for tip in df_main["payment_type"].dropna().unique()]

f_stat, p_value = stats.f_oneway(*gruplar)

print("F-istatistiği:", round(f_stat, 4))
print("P-value:", p_value)

if p_value < 0.05:
    print("H0 reddedilir — ödeme yöntemleri arasında anlamlı fark var.")
else:
    print("H0 reddedilemez — anlamlı bir fark olduğuna dair yeterli kanıt yok.")

F-istatistiği: 8.5021
P-value: 7.438848195130616e-07
H0 reddedilir — ödeme yöntemleri arasında anlamlı fark var.


**Sonuç:** ANOVA testi, ödeme yöntemleri arasında istatistiksel olarak
anlamlı bir fark olduğunu göstermektedir (F = 8.50, p < 0.001). Ancak bu
sonucu dikkatli yorumlamak gerekir: ana ödeme yöntemlerinin (kredi kartı,
boleto, banka kartı) ortalama puanları birbirine çok yakındır (3.96–4.15
bandı) ve bu farkın **pratik önemi düşüktür.** İstatistiksel anlamlılık,
büyük örneklem büyüklüğümüzün (~99,000+ gözlem) küçük farkları bile
tespit edebilmesinden kaynaklanmaktadır — "not_defined" kategorisinin
(yalnızca 3 kayıt, 1.67 ortalama puan) genel ortalamayı çekmesi de bu
farka katkıda bulunmuş olabilir.

**İş çıkarımı:** Ödeme yöntemi seçimi, müşteri memnuniyeti üzerinde pratik
anlamda belirleyici bir faktör değildir. Platformun farklı ödeme
yöntemlerini desteklemesi, memnuniyet açısından risk taşımamaktadır.

In [ ]:
df_ana_odemeler = df_main[df_main["payment_type"] != "not_defined"]
gruplar2 = [df_ana_odemeler[df_ana_odemeler["payment_type"] == tip]["review_score"].dropna()
            for tip in df_ana_odemeler["payment_type"].dropna().unique()]
f_stat2, p_value2 = stats.f_oneway(*gruplar2)
print("not_defined hariç F-istatistiği:", round(f_stat2, 4))
print("not_defined hariç P-value:", p_value2)

not_defined hariç F-istatistiği: 8.5216
not_defined hariç P-value: 1.177858964077286e-05


**Ek Doğrulama:** `not_defined` kategorisi (yalnızca 3 kayıt) çıkarıldığında
bile sonuç neredeyse değişmemiştir (F = 8.52, p < 0.001) — yani anlamlı
farkın kaynağı bu aşırı uç kategori değil, ana ödeme yöntemleri (kredi
kartı, boleto, banka kartı, voucher) arasındaki gerçek, tutarlı bir
farktır.

**Yeniden değerlendirme:** Bu durumda istatistiksel anlamlılık daha
sağlam bir zeminde duruyor. Ancak fark büyüklüğü hâlâ pratik olarak
küçüktür (puan aralığı 3.96–4.15, yaklaşık 0.2 puanlık bir bant) — bu,
"ödeme yöntemi memnuniyeti hafifçe etkiliyor, ama iş kararlarını
değiştirecek büyüklükte bir etki değil" şeklinde yorumlanabilir.
Muhtemel açıklama: banka kartı (debit_card, 4.15) kullanıcıları,
büyük olasılıkla daha küçük/planlı alışverişler yaptıkları için
(EDA'da ortalama tutarının en düşük ikinci grup olduğunu görmüştük)
hafifçe daha yüksek memnuniyet gösteriyor olabilir.

### Ödeme Ekseni: Taksit Sayısı ile Ödeme Tutarı İlişkisi

Keşifsel analizde, taksit sayısı ile ödeme tutarı arasında zayıf-orta
düzeyde (r = 0.27) pozitif bir korelasyon gözlemlemiştik. Bu ilişkinin
istatistiksel anlamlılığını test ediyoruz.

**H0:** Taksit sayısı ile ödeme tutarı arasında ilişki yoktur.
**H1:** Taksit sayısı ile ödeme tutarı arasında ilişki vardır.

In [ ]:
veri_temiz2 = df_main[["payment_installments", "payment_value"]].dropna()

korelasyon2, p_value2 = stats.pearsonr(veri_temiz2["payment_installments"], veri_temiz2["payment_value"])

print("Korelasyon katsayısı:", round(korelasyon2, 4))
print("P-value:", p_value2)

if p_value2 < 0.05:
    print("H0 reddedilir — taksit sayısı ile ödeme tutarı arasında anlamlı bir ilişki var.")
else:
    print("H0 reddedilemez — anlamlı bir ilişki olduğuna dair yeterli kanıt yok.")

Korelasyon katsayısı: 0.2736
P-value: 0.0
H0 reddedilir — taksit sayısı ile ödeme tutarı arasında anlamlı bir ilişki var.


**Sonuç:** Taksit sayısı ile ödeme tutarı arasında istatistiksel olarak
anlamlı (p < 0.001) ve zayıf-orta düzeyde (r = 0.27) pozitif bir ilişki
bulunmuştur.

Bu sonuç, önceki iki testten (fiyat-puan ilişkisi ve ödeme yöntemi-puan
ilişkisi) farklı bir nitelik taşıyor: orada istatistiksel anlamlılık
olsa bile korelasyon/fark büyüklüğü pratik olarak ihmal edilebilir
düzeydeydi. Burada ise korelasyon katsayısı (0.27) gerçek bir "zayıf-orta"
ilişki seviyesinde — yani hem istatistiksel olarak hem pratik olarak
anlamlı bir bulgu.

**İş çıkarımı (doğrulanmış):** Taksit imkanı sunmak, müşterilerin daha
yüksek tutarlı alışverişler yapmasını teşvik ediyor görünmektedir. Bu,
checkout sürecinde taksit seçeneklerinin daha görünür/vurgulu sunulmasının
(örn. "12 taksitle X TL" gibi mesajlaşma), ortalama sepet tutarını
artırabilecek, veriyle desteklenmiş bir öneri olduğunu göstermektedir.

### Satıcı Ekseni: Teslimat Süresi ile Review Puanı İlişkisi

Satıcı ekseninde, en yavaş teslimat yapan satıcının en düşük review
puanını aldığını gözlemlemiştik. Bu ilişkinin genel veri setinde de
geçerli olup olmadığını test ediyoruz.

**H0:** Teslimat süresi ile review puanı arasında ilişki yoktur.
**H1:** Teslimat süresi ile review puanı arasında ilişki vardır (muhtemelen negatif — süre arttıkça puan azalır).

In [ ]:
veri_temiz3 = df_main[["teslimat_suresi", "review_score"]].dropna()

korelasyon3, p_value3 = stats.pearsonr(veri_temiz3["teslimat_suresi"], veri_temiz3["review_score"])

print("Korelasyon katsayısı:", round(korelasyon3, 4))
print("P-value:", p_value3)

if p_value3 < 0.05:
    print("H0 reddedilir — teslimat süresi ile review puanı arasında anlamlı bir ilişki var.")
else:
    print("H0 reddedilemez — anlamlı bir ilişki olduğuna dair yeterli kanıt yok.")

Korelasyon katsayısı: -0.3027
P-value: 0.0
H0 reddedilir — teslimat süresi ile review puanı arasında anlamlı bir ilişki var.


**Sonuç:** Teslimat süresi ile review puanı arasında istatistiksel olarak
anlamlı (p < 0.001) ve orta düzeyde (r = -0.30) **negatif** bir ilişki
bulunmuştur. Bu, dört testimiz arasında en güçlü korelasyon katsayısına
sahip olanı ve satıcı ekseninde gözlemlediğimiz örüntüyü (yavaş teslimat
yapan satıcının düşük puan alması) tüm veri setinde doğrulamaktadır.

**İş çıkarımı:** Teslimat süresi, müşteri memnuniyetini etkileyen en
somut, en güçlü ve en doğrudan aksiyona dönüştürülebilir faktördür.
Lojistik/kargo süreçlerinin iyileştirilmesi (özellikle 15+ gün teslimat
süresi olan satıcılarla çalışılması), platformun genel memnuniyet
seviyesini yükseltmek için en yüksek potansiyelli müdahale alanıdır.

**İstatistiksel Testler Özeti:**
En güçlü ve en aksiyona dönük bulgu, teslimat süresinin memnuniyet
üzerindeki negatif etkisidir (r = -0.30). Taksit imkanının harcama
tutarını artırması (r = 0.27) ikinci önemli bulgu. Fiyat ve ödeme
yönteminin memnuniyet üzerinde pratik bir etkisi bulunmamıştır.

### Ek İnceleme: En Yüksek Puanlı Ürünler Neden Bu Kadar Beğeniliyor?

Görselleştirmeye geçmeden önce, en yüksek puanlı 10 ürünün öne çıkan
özelliklerine (fiyat, sipariş sayısı, ortalama teslimat süresi) hızlıca
bakarak, yüksek memnuniyetin arkasındaki olası sebepleri araştırıyoruz.

In [ ]:
top10_puan_id = en_yuksek_puan.index

detay_yuksek_puan = df_main[df_main["product_id"].isin(top10_puan_id)].groupby("product_id").agg({
    "price": "mean",
    "teslimat_suresi": "mean",
    "order_id": "nunique"
}).round(2)

detay_yuksek_puan.columns = ["ortalama_fiyat", "ortalama_teslimat_gun", "siparis_sayisi"]
display(detay_yuksek_puan)

print("\nKarşılaştırma için genel ortalamalar:")
print("Genel ortalama fiyat:", round(df_urun_gelir["price"].mean(), 2))
print("Genel ortalama teslimat süresi:", round(df_main["teslimat_suresi"].mean(), 2))

,ortalama_fiyat,ortalama_teslimat_gun,siparis_sayisi
product_id,,,
00066f42aeeb9f3007548bb9d3f33c38,101.65,17.0,1
000b8f95fcb9e0096488278317764d19,58.90,6.0,2
000d9be29b5207b54e86aa1b1ac54872,199.00,7.0,1
8aebbc3445bef4a356ae41d226a7b5da,119.90,17.0,1
8aeef27d525d6bfa3b48e599a6c15ffd,49.95,17.0,1
ffd63ee42a5c8cc5a15a1c8e2aa50011,77.00,7.0,1
ffd7628b0b0b98ebc549e8e4c54a59af,79.90,6.0,1
ffd9ac56db9194a413298faaa03cd176,89.00,6.0,1
fff9553ac224cec9d15d49f5a263411f,32.00,10.0,1



Karşılaştırma için genel ortalamalar:
Genel ortalama fiyat: 120.65
Genel ortalama teslimat süresi: 12.02


### Kritik Ek Bulgu: En Yüksek Puanlı Ürünler Güvenilir mi?

En yüksek puanlı 10 ürünün sipariş sayılarına bakıldığında, bu ürünlerin
büyük çoğunluğunun (9/10) yalnızca **1 kez** sipariş edildiği görülmüştür.
Bu, önceki bulgumuzu (yüksek puanlı ürünlerin ticari başarıyla örtüşmemesi)
yeniden değerlendirmemizi gerektiriyor:

**Revize edilmiş yorum:** Bu ürünlerin yüksek ortalama puanı, gerçek bir
"gizli kalmış kalite" göstergesi olmaktan çok, **küçük örneklem etkisidir**
— tek bir siparişin puanı, doğrudan ürünün "ortalaması" haline geliyor ve
bu, istatistiksel olarak güvenilir bir sinyal değildir. Bir ürünün gerçekten
yüksek ve güvenilir bir memnuniyet gösterdiğini söyleyebilmek için, yeterli
sayıda siparişe (örneğin 10+) sahip olması gerekir.

**Düzeltilmiş fırsat değerlendirmesi:** "En yüksek puanlı ama az bilinen
ürünleri öne çıkarma" önerisi yerine, daha sağlam bir yaklaşım şudur:
**yeterli sipariş hacmine sahip (örn. 10+ sipariş) VE yüksek puanlı**
ürünleri belirleyip, bunları pazarlama/görünürlük stratejisinde önceliklendirmek.

In [ ]:
# Platformdaki ürün başına ortalama sipariş sayısı
ortalama_siparis_sayisi = urun_siparis_sayisi.mean()
print("Ürün başına ortalama sipariş sayısı:", round(ortalama_siparis_sayisi, 2))

# Bu ortalamanın üzerinde sipariş almış ürünleri filtrele
temsili_urunler = urun_siparis_sayisi[urun_siparis_sayisi >= ortalama_siparis_sayisi].index

# Bu ürünler arasında en yüksek puanlıları (4.85 ve üzeri) bul
guvenilir_yuksek_puan_v2 = df_main[df_main["product_id"].isin(temsili_urunler)].groupby("product_id")["review_score"].mean()
guvenilir_yuksek_puan_v2 = guvenilir_yuksek_puan_v2[guvenilir_yuksek_puan_v2 >= 4.85].sort_values(ascending=False)

display(guvenilir_yuksek_puan_v2)
print("\nBu kritere uyan ürün sayısı:", len(guvenilir_yuksek_puan_v2))

Ürün başına ortalama sipariş sayısı: 3.11


,review_score
product_id,
fffdb2d0ec8d6a61f0a0a0db3f25b441,5.000000
007c63ae4b346920756b5adcad8095de,5.000000
00905d58c87afcbce21420b3712cacaa,5.000000
f832d1b20241274bc51c2d691b0f4b94,5.000000
f81a1edab43cbf08299503316612bacf,5.000000
...,...
12fc9ab82dd45f3824881d94f79edb38,4.857143
1782400950423c9b12600278b8ef65d3,4.857143
195a72beccf48da20d71e2d7530b470a,4.857143



Bu kritere uyan ürün sayısı: 451


In [ ]:
print("Ürün başına sipariş sayısı - özet istatistikler:")
print(urun_siparis_sayisi.describe())

print("\nMedyan:", urun_siparis_sayisi.median())

Ürün başına sipariş sayısı - özet istatistikler:
count    32951.000000
mean         3.108403
std          9.456937
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        467.000000
Name: order_id, dtype: float64

Medyan: 1.0


In [ ]:
temsili_urunler_final = urun_siparis_sayisi[urun_siparis_sayisi >= 20].index
print("En az 20 sipariş almış ürün sayısı:", len(temsili_urunler_final))

guvenilir_son = df_main[df_main["product_id"].isin(temsili_urunler_final)].groupby("product_id")["review_score"].mean().sort_values(ascending=False)
display(guvenilir_son.head(10))

En az 20 sipariş almış ürün sayısı: 618


,review_score
product_id,
3e4176d545618ed02f382a3057de32b4,4.958333
e7f85e7f0203b7b95cc1b4c21b4b070c,4.869565
11250b0d4b709fee92441c5f34122aed,4.863636
73326828aa5efe1ba096223de496f596,4.839286
574597aaf385996112490308e37399ce,4.833333
f7f59e6186e10983a061ac7bdb3494d6,4.829268
990d135e28e075648cb7d83198fdccf4,4.814815
e7d5464b94c9a5963f7c686fc80145ad,4.750000
e7d26dd6742baca292020c158e6720c3,4.740741


**Nihai Sonuç:** En az 20 sipariş almış 618 ürün arasında en yüksek puanlı
10 ürün incelendiğinde, puanlar 4.74–4.96 aralığında, istatistiksel olarak
çok daha güvenilir bir dağılım göstermektedir (tümü 5.0 değil). Bu liste,
gerçekten yüksek ve tutarlı memnuniyet sağlayan, yeterli sipariş hacmiyle
doğrulanmış ürünleri temsil etmektedir.

**Fırsat değerlendirmesi (revize, son hali):** Bu 618 ürün — hem yeterli
talep görmüş hem de istikrarlı şekilde yüksek puan almış — platformun
gerçek "gizli şampiyonları" olarak değerlendirilebilir. Bu ürünlerin
pazarlama/görünürlük stratejilerinde önceliklendirilmesi, güvenilir bir
veriye dayanan, düşük riskli bir büyüme fırsatıdır.

## 7. GÖRSELLEŞTİRME

Bu bölümde EDA ve istatistiksel test bulgularının en güçlü olanları
Plotly ile görselleştirilmiştir.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Hacim ve gelir verilerini kategori ile birlikte hazırlayalım
satan_df = en_cok_satan.head(10).reset_index()
satan_df.columns = ["product_id", "siparis_sayisi"]
satan_df["kategori"] = satan_df["product_id"].map(urun_kategori)
satan_df["etiket"] = satan_df["kategori"] + " (" + (satan_df.index + 1).astype(str) + ")"

gelir_df = en_cok_gelir.head(10).reset_index()
gelir_df.columns = ["product_id", "toplam_gelir"]
gelir_df["kategori"] = gelir_df["product_id"].map(urun_kategori)
gelir_df["etiket"] = gelir_df["kategori"] + " (" + (gelir_df.index + 1).astype(str) + ")"

# Yan yana iki alt grafik oluştur
fig = make_subplots(rows=1, cols=2, subplot_titles=("En Çok Satan 10 Ürün (Sipariş Sayısı)",
                                                       "En Çok Gelir Getiren 10 Ürün (Toplam Gelir)"))

fig.add_trace(go.Bar(x=satan_df["etiket"], y=satan_df["siparis_sayisi"],
                      name="Sipariş Sayısı", marker_color="steelblue"), row=1, col=1)

fig.add_trace(go.Bar(x=gelir_df["etiket"], y=gelir_df["toplam_gelir"],
                      name="Toplam Gelir", marker_color="darkorange"), row=1, col=2)

fig.update_layout(title_text="Ürün Ekseni: Satış Hacmi vs Gelir Karşılaştırması", showlegend=False)
fig.show()

**Kategori Bazlı İçgörü:** Kategori isimleri incelendiğinde, hacim ve gelir
liderliğinin farklı ürün segmentlerinden geldiği görülüyor:

- **Hacim lideri kategoriler:** Ev/yaşam odaklı (cama_mesa_banho,
  moveis_decoracao, ferramentas_jardim) — sık tekrarlanan, günlük ihtiyaç
  ürünleri.
- **Gelir lideri kategoriler:** Güzellik/sağlık (beleza_saude — listenin
  ilk iki sırasında) ve teknoloji (pcs, informatica_acessorios) — yüksek
  birim fiyatlı, daha nadir alınan ürünler.

**İş çıkarımı:** Platformun pazarlama ve stok stratejisi bu iki segment
için farklılaştırılmalıdır — ev/yaşam kategorisinde erişim ve tekrar
alışverişi teşvik eden kampanyalar, güzellik/sağlık ve teknoloji
kategorisinde ise güven inşa eden (detaylı ürün açıklaması, kolay iade,
müşteri yorumları öne çıkarma gibi) yaklaşımlar daha etkili olabilir.

In [ ]:
guvenilir_df = guvenilir_son.head(10).reset_index()
guvenilir_df.columns = ["product_id", "ortalama_puan"]
guvenilir_df["kategori"] = guvenilir_df["product_id"].map(urun_kategori)
guvenilir_df["etiket"] = guvenilir_df["kategori"] + " (" + (guvenilir_df.index + 1).astype(str) + ")"

fig3 = px.bar(guvenilir_df, x="etiket", y="ortalama_puan",
              title="En Az 20 Sipariş Almış, En Yüksek Puanlı 10 Ürün (Kategoriye Göre)",
              range_y=[4, 5],
              hover_data=["product_id"])
fig3.show()

In [ ]:
ornek = df_main[["payment_installments", "payment_value"]].dropna().sample(2000, random_state=42)

fig4 = px.scatter(ornek, x="payment_installments", y="payment_value",
                   title="Taksit Sayısı ile Ödeme Tutarı İlişkisi (r=0.27)",
                   trendline="ols",
                   labels={"payment_installments": "Taksit Sayısı", "payment_value": "Ödeme Tutarı"})
fig4.show()

In [ ]:
odeme_df = df_payments["payment_type"].value_counts().reset_index()
odeme_df.columns = ["odeme_yontemi", "sayi"]

fig6 = px.bar(odeme_df, x="odeme_yontemi", y="sayi",
              title="Ödeme Yöntemi Dağılımı")
fig6.show()

In [ ]:
df_main["teslimat_grubu"] = pd.cut(df_main["teslimat_suresi"],
                                      bins=[0, 5, 10, 15, 20, 100],
                                      labels=["0-5 gün", "5-10 gün", "10-15 gün", "15-20 gün", "20+ gün"])

fig5b = px.box(df_main.dropna(subset=["teslimat_grubu", "review_score"]),
               x="teslimat_grubu", y="review_score",
               title="Teslimat Süresi Aralığına Göre Review Puanı Dağılımı (r=-0.30)")
fig5b.show()

In [ ]:
satici_df = satici_performans_gelir.reset_index()
satici_df["etiket"] = "Satıcı " + (satici_df.index + 1).astype(str)

fig7b = px.bar(satici_df, x="etiket", y="ort_review_puani",
               title="Top 10 Gelirli Satıcının Ortalama Review Puanı",
               hover_data=["seller_id", "ort_teslimat_gun"])
fig7b.show()

**Görselleştirme Özeti:** Yukarıdaki grafikler, EDA ve istatistiksel test
bulgularının en güçlü olanlarını görsel olarak özetlemektedir: ürün
segmentasyonu (hacim vs değer), güvenilir yüksek performanslı ürünler,
teslimat süresinin memnuniyet üzerindeki etkisi, ödeme yöntemi tercihleri
ve satıcı performans farklılıkları.

## 8. SONUÇ VE ÖNERİLER

Bu çalışma, Olist Brezilya e-ticaret veri setini kullanarak müşteri,
sipariş, ürün ve ödeme ilişkilerini incelemiş; üç eksende (ürün, ödeme,
satıcı) yapılan keşifsel analiz ve dört istatistiksel test sonucunda
aşağıdaki temel bulgulara ulaşılmıştır:

**1. Ürün Segmentasyonu:** Platformdaki ürünler iki farklı profile ayrılıyor
— hacim odaklı (ev/yaşam kategorisi, düşük fiyat, yüksek sipariş sayısı) ve
değer odaklı (güzellik/sağlık ve teknoloji, yüksek fiyat, düşük sipariş
sayısı). Bu iki segment için farklı pazarlama ve stok stratejileri
uygulanması önerilir.

**2. Gizli Fırsat Ürünleri:** En az 20 sipariş almış, ortalama 4.7+ puanlı
618 ürün belirlenmiştir — bunlar hem talep görmüş hem yüksek memnuniyet
sağlayan, ancak mevcut ticari öncelik listelerinde (satış/gelir liderliği)
yer almayan ürünlerdir. Bu ürünlerin görünürlüğünün artırılması, düşük
riskli bir büyüme fırsatıdır.

**3. Teslimat Süresi — En Güçlü Aksiyon Alanı:** Teslimat süresi ile
müşteri memnuniyeti arasında istatistiksel olarak anlamlı ve orta düzeyde
negatif bir ilişki bulunmuştur (r=-0.30, p<0.001). Bu, dört test arasında
en güçlü ve en doğrudan aksiyona dönüştürülebilir bulgudur — 15+ gün
teslimat süresi olan satıcılarla lojistik iyileştirmesi önceliklendirilmelidir.

**4. Taksit İmkanının Etkisi:** Taksit sayısı ile ödeme tutarı arasında
orta düzeyde pozitif bir ilişki bulunmuştur (r=0.27, p<0.001) — taksit
seçeneğinin checkout sürecinde daha görünür kılınması, ortalama sepet
tutarını artırabilir.

**5. Ödeme Yöntemi ve Fiyatın Sınırlı Etkisi:** Ödeme yöntemi ve ürün
fiyatının müşteri memnuniyeti üzerinde pratik olarak anlamlı bir etkisi
bulunmamıştır — bu alanlarda aksiyon önceliği düşüktür.

**Genel Değerlendirme:** Analiz, platformun büyüme potansiyelinin ürün
kataloğunun genişletilmesinden çok, **mevcut varlıkların (yüksek puanlı
ürünler, hızlı satıcılar) daha iyi görünür kılınmasında ve lojistik
performansın iyileştirilmesinde** yattığını göstermektedir.

## Ek İnceleme: Müşteri/Bölge Ekseni

Ana analiz ürün, ödeme ve satıcı eksenlerine odaklanmıştır. Zaman kısıtı
nedeniyle kapsam dışı bırakılan müşteri verisini (df_customers) kısaca
incelemek, ek bir bakış açısı sunmaktadır: hangi eyaletler en çok sipariş
veriyor ve ortalama harcama bölgeye göre nasıl değişiyor?

Bu inceleme, ana bulgularımızı tamamlayan, ancak derinlemesine analiz
edilmemiş bir ek gözlemdir.

In [ ]:
eyalet_analiz = df_main.groupby("customer_state").agg({
    "order_id": "nunique",
    "payment_value": "mean"
}).round(2).sort_values("order_id", ascending=False)

eyalet_analiz.columns = ["siparis_sayisi", "ortalama_odeme"]
display(eyalet_analiz.head(10))

,siparis_sayisi,ortalama_odeme
customer_state,,
SP,41746,153.72
RJ,12852,180.15
MG,11635,170.14
RS,5466,176.51
PR,5045,178.69
SC,3637,184.41
BA,3380,196.79
DF,2140,174.12
ES,2033,173.14


**Bulgu:** SP (São Paulo), sipariş hacminde açık ara lider (41,746 sipariş
— ikinci sıradaki RJ'nin ~3.2 katı), ancak ortalama sipariş tutarı
(153.72) listedeki en düşük değerlerden biridir. Buna karşılık GO
(Goiás), çok daha düşük hacimle (2,020 sipariş) en yüksek ortalama
ödemeye (211.06) sahiptir.

**Genel Tema:** Bu bulgu, çalışma boyunca tekrar eden bir örüntüyü
coğrafi düzeyde de doğrulamaktadır: platformda **hacim ve değer
sistematik olarak farklı segmentlerden gelmektedir** — bu, ürünlerde
(en çok satan vs en çok kazandıran), satıcılarda ve şimdi coğrafi
bölgelerde tutarlı şekilde gözlemlenmiştir.

**İş çıkarımı:** SP'deki yüksek hacim, muhtemelen büyük şehir
yaşamının getirdiği sık ama küçük tutarlı alışverişleri yansıtıyor;
GO gibi bölgelerdeki yüksek ortalama tutar ise daha az ama daha
büyük/planlı alışverişlere işaret edebilir. Bölgesel pazarlama
stratejisi bu farkı göz önünde bulundurmalıdır — SP'de erişim/hacim
odaklı, GO gibi bölgelerde ise değer/güven odaklı yaklaşımlar
daha uygun olabilir.